# Pratica — Lezione 9. Pianificazione e modelli

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

In questa pratica programmi tu stesso il cuore di Dyna-Q: il passo di pianificazione simulata. Poi misuri l'effetto reale di variare quanti passi di pianificazione fa l'agente per ogni passo reale.

Usa lo stesso mondo a griglia con muri della lezione: sei righe, nove colonne, due segmenti di parete, partenza nell'angolo in basso a sinistra e obiettivo nell'angolo in alto a destra.

**Prima di iniziare, esegui le due celle seguenti per avere pronta la struttura dell'ambiente.** Non serve modificarle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


In [ ]:
# --- Mundo-cuadrícula (idéntico al de la clase) ---
ALTO, ANCHO = 6, 9
INICIO = (5, 0)
META = (0, 8)
PAREDES = {(1, 3), (2, 3), (3, 3), (0, 6), (1, 6), (2, 6)}
ACCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # arriba, abajo, izquierda, derecha
N_ACCIONES = len(ACCIONES)


def paso_entorno(estado, accion_idx):
    """Aplica una acción de forma determinista: si el movimiento choca contra
    un muro o el borde del mundo, el agente se queda donde estaba."""
    if estado == META:
        return estado, 0.0, True
    d_fila, d_col = ACCIONES[accion_idx]
    fila, col = estado
    nueva_fila = min(max(fila + d_fila, 0), ALTO - 1)
    nueva_col = min(max(col + d_col, 0), ANCHO - 1)
    nuevo_estado = (nueva_fila, nueva_col)
    if nuevo_estado in PAREDES:
        nuevo_estado = estado
    terminado = nuevo_estado == META
    recompensa = 0.0 if terminado else -1.0
    return nuevo_estado, recompensa, terminado


def indice(estado):
    """Convierte una casilla (fila, columna) en un índice único de fila en la tabla Q."""
    return estado[0] * ANCHO + estado[1]


def elegir_accion(Q, estado, epsilon):
    """Política epsilon-greedy, la misma que ya conoces de la clase tres."""
    if np.random.random() < epsilon:
        return np.random.randint(N_ACCIONES)
    fila_q = Q[indice(estado)]
    mejores = np.flatnonzero(fila_q == np.max(fila_q))
    return np.random.choice(mejores)


## Esercizio 1 — Completa la pianificazione simulata di Dyna-Q

In classe hai visto che Dyna-Q ha due fasi in ogni passo: apprendimento diretto (con esperienza reale) e pianificazione simulata (con esperienza immaginata dal modello). Qui sotto è già costruita la fase uno completa, insieme al `modelo`, un dizionario che memorizza ogni transizione reale vissuta dall'agente: `(stato, azione) -> (stato_successivo, ricompensa)`.

**Il tuo compito è completare la fase due, dove c'è `# TODO`.** Dentro il ciclo `for _ in range(n_planning)`, devi:

1. Scegliere a caso una coppia stato-azione già salvata in `modelo` (suggerimento: converti `modelo.items()` in una lista con `list(...)` e scegli un indice a caso con `np.random.randint`)
2. Leggere da quella voce lo stato, l'azione, lo stato successivo simulato e la ricompensa simulata
3. Calcolare l'obiettivo di Q-learning per quella transizione immaginata, come nella fase uno ma senza richiamare di nuovo `paso_entorno`
4. Aggiornare `Q` con quell'obiettivo, con la stessa formula della fase uno

**Suggerimenti**

- La formula di aggiornamento è la stessa nelle due fasi: `Q[i, a] += alpha * (objetivo - Q[i, a])`
- L'obiettivo è `recompensa + gamma * np.max(Q[indice(estado_siguiente)])`
- Non serve toccare l'ambiente reale né `elegir_accion` in questa fase: tutte le informazioni vengono da `modelo`

Con il `# TODO` non completato, la cella di prova qui sotto viene eseguita senza errori, ma non noterai alcun miglioramento aumentando `n_planning`: questo è il segnale che ti manca ancora da implementare la pianificazione.

In [ ]:
def ejecutar_dyna_q_incompleto(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    """Entrena un agente Dyna-Q tabular. Con n_planning=0 equivale a Q-learning puro."""
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}                      # modelo aprendido: (estado, accion) -> (estado_siguiente, recompensa)
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            # --- Fase 1: aprendizaje directo, con experiencia real (ya completa) ---
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            # --- Fase 2: planificación simulada (te toca completarla) ---
            for _ in range(n_planning):
                # TODO: 1. Elige al azar una transición ya guardada en `modelo`
                # TODO: 2. Extrae estado, acción, estado_siguiente y recompensa simulados
                # TODO: 3. Calcula el objetivo de Q-learning para esa transición imaginada
                # TODO: 4. Actualiza Q con ese objetivo, igual que en la fase 1
                pass

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio


**Esegui la cella seguente per provare la tua implementazione.** Confronta l'ultimo episodio di un agente senza pianificazione (`n_planning=0`) con uno che fa 20 passi di pianificazione (`n_planning=20`): se il tuo codice funziona, il secondo dovrebbe impiegare parecchi passi in meno per raggiungere l'obiettivo.

In [ ]:
resultado_sin_planificar = ejecutar_dyna_q_incompleto(0, 30)
resultado_con_planificar = ejecutar_dyna_q_incompleto(20, 30)

print("Pasos en el último episodio, sin planificación:", resultado_sin_planificar[-1])
print("Pasos en el último episodio, con 20 pasos de planificación:", resultado_con_planificar[-1])


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def ejecutar_dyna_q_incompleto(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    """Addestra un agente Dyna-Q tabulare. Con n_planning=0 equivale al Q-learning puro."""
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}                      # modello appreso: (stato, azione) -> (stato_successivo, ricompensa)
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            # --- Fase 1: apprendimento diretto, con esperienza reale ---
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            # --- Fase 2: pianificazione simulata ---
            transiciones_vistas = list(modelo.items())
            for _ in range(n_planning):
                # 1. Scegliamo a caso una transizione già osservata
                (s_sim, a_sim), (s_sig_sim, r_sim) = transiciones_vistas[
                    np.random.randint(len(transiciones_vistas))
                ]
                # 2 e 3. Calcoliamo l'obiettivo di Q-learning su quella transizione immaginata
                i_sim, i_sig_sim = indice(s_sim), indice(s_sig_sim)
                objetivo_sim = r_sim + gamma * np.max(Q[i_sig_sim])
                # 4. Aggiorniamo Q con la stessa formula della fase 1
                Q[i_sim, a_sim] += alpha * (objetivo_sim - Q[i_sim, a_sim])

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio
```
</details>

## Esercizio 2 — Confronta la velocità di apprendimento in base ai passi di pianificazione

Ora che hai Dyna-Q completo, tocca a te misurarne l'effetto reale. Addestrerai l'agente con diversi valori di `n_planning`, farai la media di più ripetizioni per attenuare il caso e tracerai quanti passi serve ogni configurazione per raggiungere l'obiettivo in ciascun episodio.

**Perché tu possa andare avanti anche se la tua soluzione dell'esercizio uno non è ancora pronta**, questa cella usa `ejecutar_dyna_q`, una copia già completa e funzionante della funzione Dyna-Q.

**Il tuo compito è completare il ciclo sperimentale**, dove c'è `# TODO`:

1. Scorri ogni valore di `n_planning` in `configuraciones`
2. Per ciascuno, ripeti l'addestramento `n_repeticiones` volte, accumulando i passi per episodio di ogni ripetizione
3. Salva in `resultados[n_planning]` la media di quelle ripetizioni

**Suggerimenti**

- Usa `np.zeros(n_episodios)` per inizializzare l'accumulatore di ogni configurazione, e sommagli il risultato di ogni ripetizione
- Alla fine dividi per `n_repeticiones` per ottenere la media
- La cella del grafico è già completa: appena riempi `resultados`, eseguila così com'è

In [ ]:
# Copia completa de Dyna-Q, lista para usar en este ejercicio
def ejecutar_dyna_q(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            transiciones_vistas = list(modelo.items())
            for _ in range(n_planning):
                (s_sim, a_sim), (s_sig_sim, r_sim) = transiciones_vistas[
                    np.random.randint(len(transiciones_vistas))
                ]
                i_sim, i_sig_sim = indice(s_sim), indice(s_sig_sim)
                objetivo_sim = r_sim + gamma * np.max(Q[i_sig_sim])
                Q[i_sim, a_sim] += alpha * (objetivo_sim - Q[i_sim, a_sim])

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio


# --- Bucle experimental: te toca completarlo ---
n_episodios = 40
n_repeticiones = 5
configuraciones = [0, 5, 50]
resultados = {}

# TODO: recorre `configuraciones`, entrena `n_repeticiones` veces con cada
# valor de n_planning, acumula los pasos por episodio y guarda en
# `resultados[n_planning]` el promedio de las repeticiones.


In [ ]:
# --- Gráfica de la comparación (ya completa, no hace falta tocarla) ---
plt.figure(figsize=(10, 5))
for n_planning in configuraciones:
    etiqueta = "Q-learning puro (0 pasos)" if n_planning == 0 else f"Dyna-Q ({n_planning} pasos de planificación)"
    plt.plot(resultados[n_planning], label=etiqueta)
plt.xlabel("Episodio")
plt.ylabel("Pasos hasta llegar a la meta")
plt.title("Efecto del número de pasos de planificación")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
n_episodios = 40
n_repeticiones = 5
configuraciones = [0, 5, 50]
resultados = {}

for n_planning in configuraciones:
    acumulado = np.zeros(n_episodios)
    for _ in range(n_repeticiones):
        acumulado += ejecutar_dyna_q(n_planning, n_episodios)
    resultados[n_planning] = acumulado / n_repeticiones
```
</details>

## Per continuare a esercitarti

**Cambia `configuraciones` in `[1, 10, 100]` e riesegui l'esperimento completo.** A partire da quanti passi di pianificazione smette di vedersi un miglioramento chiaro rispetto al costo extra di calcolo? Discutine con il tuo team.